<a href="https://colab.research.google.com/github/andressonsino/sistema-experto/blob/main/sistema_experto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Desafío Individual: Sistema de Gestión de Obra Inteligente

## Contexto del Problema
Una empresa constructora está desarrollando una torre de gran altura y necesita automatizar dos procesos críticos para garantizar la seguridad y la eficiencia operativa:
* Evaluación de Riesgos en Obra (Lógica Deductiva): Determinar si es seguro continuar con las tareas de altura o excavación basándose en sensores climáticos y estructurales.
* Planificación de Maquinaria Pesada (Satisfacción de Restricciones): Asignar equipos (grúas, excavadoras) a zonas específicas del predio respetando límites de seguridad y espacio físico.

**Misión A:** Diagnóstico de Seguridad con expertaDebes programar un motor de inferencia que reciba datos de sensores y devuelva el nivel de riesgo del sitio. Este sistema actúa como un Cerebro Lógico para evitar accidentes.

Reglas a Implementar:
* **Riesgo Crítico** (Paro de Obra):  Si la velocidad del viento es $> 60\ km/h$ o si se detectan grietas en el suelo de fundación.
* **Riesgo Moderado** (Precaución): Si la velocidad del viento está entre $40\ km/h$ y $60\ km/h$ o si hay humedad extrema en zonas de excavación.
* **Bajo Riesgo** (Operación Normal): Si los vientos son $< 40\ km/h$ y no hay alertas estructurales activas.

**Consigna Técnica:** Utiliza el parámetro salience para asegurar que la regla de Riesgo Crítico se evalúe con la máxima prioridad ante cualquier otra condición.El sistema debe imprimir el diagnóstico final y la orden de seguridad correspondiente.

### Mision A

In [ ]:
# Instalación
!pip install experta --quiet
!pip install python-constraint --quiet

# Parche compatibilidad con Python 3.12 (Google Colab)
!sed -i 's/collections.Mapping/collections.abc.Mapping/' /usr/local/lib/python3.12/dist-packages/frozendict/__init__.py

print("✅ Librerías instaladas y parche aplicado correctamente.")

  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.66 requires frozendict>=2.3.4, but you have frozendict 1.2 which is incompatible.
  Preparing metadata (setup.py) ... done
✅ Librerías instaladas y parche aplicado correctamente.


In [ ]:
from experta import *
class VelocidadViento(Fact):
    pass
class GrietasSuelo(Fact):
    pass
class HumedadExtrema(Fact):
    pass
class AlertasEstructurales(Fact):
    pass

class MotorSeguridad(KnowledgeEngine):
    # TODO: Implementar las reglas de seguridad aquí
    @Rule(OR(
        VelocidadViento(valor=P(lambda x: x > 60)),
        GrietasSuelo(detectada=L(True))
        ), salience=100)
    def riesgo_critico(self):
        print("🔴 RIESGO CRÍTICO - PARO DE OBRA")
        self.halt()
    @Rule(OR(
        VelocidadViento(valor=P(lambda x: 40 <= x <= 60)),
        HumedadExtrema(detectada=L(True))
        ), salience=50)
    def riesgo_moderado(self):
        print("🟠 RIESGO MODERADO - PRECAUCIÓN")
        self.halt()
    @Rule(AND(
        VelocidadViento(valor=P(lambda x: x < 40)),
        AlertasEstructurales(activa=L(False))
        ))
    def riesgo_bajo(self):
        print("🟢 BAJO RIESGO - OPERACIÓN NORMAL")
        self.halt()

# Prueba el sistema con viento de 65 km/h y presencia de grietas
motor = MotorSeguridad()
motor.reset()
motor.declare(VelocidadViento(valor=65), GrietasSuelo(detectada=True), HumedadExtrema(detectada=False), AlertasEstructurales(activa=False,))
motor.run()

🔴 RIESGO CRÍTICO - PARO DE OBRA


### Uso de `salience` y `halt()`

- **`salience`**: define la prioridad de cada regla. A mayor valor, más urgente es la regla.

- **`halt()`**: detiene el motor de inferencia inmediatamente después de ejecutar la acción. Lo usamos para que solo se muestre el nivel de riesgo más alto y no se impriman mensajes innecesarios. Sin `halt()`, el sistema podría mostrar dos niveles distintos si las condiciones se solapan, lo cual sería inconsistente en un contexto de seguridad.

**Resolución misión A**

**Justificación funcional:**

Para la Misión A, el modelo de Motor de Inferencia (experta) es superior a los condicionales if/else porque ofrece mayor escalabilidad y una gestión automática de prioridades. Mientras que un bloque de if/else se vuelve rígido, difícil de mantener y propenso a errores al agregar nuevas reglas o sensores (creando un "código espagueti"), experta permite un enfoque modular y declarativo. Cada regla de seguridad es independiente; el sistema evalúa automáticamente las prioridades matemáticas (como el salience para emergencias), garantizando que las situaciones críticas se atiendan primero sin necesidad de ordenar manualmente el código fuente, separando de forma limpia los datos de los sensores de la lógica de decisión.

**Mi realización del trabajo:**
1.   Decidí modelar un Fact por sensor, lo cual es la forma más modular y limpia, identifiqué 4 sensores.
2.   Cree la clase MotorSeguridad al cual le definí las 3 reglas con sus condiciones.
3.   Implementé la Regla de Riesgo Crítico (Máxima Prioridad). Se activa si el valor del viento es estrictamente mayor a 60 o si se detectan grietas en el suelo
Le asigné un salience=100 para garantizar que el motor evalúe esta condición de emergencia antes que cualquier otra.
4.  Implementé la Regla de Riesgo Moderado. Se activa si el viento está en el rango entre 40 y 60 (inclusive) o si se detecta humedad extrema.
iene un salience=50, por lo que solo se ejecutará si la regla crítica no se cumplió.
5.  Implementé la Regla de Riesgo Bajo que exige que se cumplan estas condiciones simultáneamente: el viento debe ser menor a 40 y no debe haber alertas estructurales activas.
Al no tener salience definido, toma el valor por defecto (0), siendo la última en evaluarse.


**Misión B:** Ubicación de Equipos con python-constraint

Debes encontrar la distribución óptima de 3 máquinas pesadas en 3 zonas de trabajo distintas. El sistema debe "podar" las opciones que violen las normativas de seguridad.

Restricciones (Reglas de Oro):
* **Grúa Torre:** Solo puede ubicarse en la Zona_Estable (debido a la necesidad de una base de hormigón reforzada).
* **Excavadora:** No puede ingresar a la Zona_Estrecha debido a sus dimensiones.
* **Hormigonera:** No puede estar en la misma zona que la Grúa Torre para evitar congestión de camiones.
* **Exclusividad:** Cada zona solo puede albergar una máquina a la vez para evitar colisiones.Consigna Técnica:Define las variables (Máquinas) y el dominio (Zonas de la obra).

Aplica las funciones de restricción para que el motor de búsqueda encuentre la única configuración válida.

In [ ]:
from constraint import *

def planificar_maquinaria():
    problem = Problem()
    # TODO: Definir máquinas (variables) y zonas (dominios)
    problem.addVariable("grua_torre", ["zona_estable", "zona_estrecha", "zona_norte"])
    problem.addVariable("excavadora", ["zona_estable", "zona_estrecha", "zona_norte"])
    problem.addVariable("hormigonera", ["zona_estable", "zona_estrecha", "zona_norte"])
    problem.addConstraint(lambda g: g == "zona_estable", ["grua_torre"])
    problem.addConstraint(lambda e: e != "zona_estrecha", ["excavadora"])
    problem.addConstraint(lambda h, g :h !=g, ["hormigonera", "grua_torre"])
    # problem.addConstraint(lambda a: a == "", [""])
    problem.addConstraint(AllDifferentConstraint())
    # TODO: Aplicar restricciones lógicas
    soluciones = problem.getSolutions()
    print(f"Configuraciones seguras encontradas: {soluciones}")

planificar_maquinaria()

Configuraciones seguras encontradas: [{'grua_torre': 'zona_estable', 'hormigonera': 'zona_estrecha', 'excavadora': 'zona_norte'}]


**Resolución misión B**

**Justificación funcional:**

Para la Misión B, el modelo de Grafos y Restricciones (python-constraint) supera a los bucles y condicionales tradicionales por su eficiencia matemática frente a la explosión combinatoria. En lugar de usar múltiples for e if para generar y filtrar todas las combinaciones posibles (lo cual colapsaría en escenarios reales con más variables), el motor de restricciones "poda" inmediatamente las opciones inválidas en el árbol de búsqueda (por ej. descartando ramas enteras si la Grúa no está en la zona correcta). Esto transforma un problema imperativo complejo en uno declarativo simple: se definen las "reglas de oro" matemáticas (exclusividad, posiciones no válidas) y el algoritmo subyacente encuentra la intersección válida de forma rápida y eficiente.

**Mi realización del trabajo:**
1.  Definí las cosas que quiero ubicar, en este caso las 3 máquinas, con sus valores posibles para cada una, usé el metodo **addVariable()**.
2.  Definí la restricción 1, **Grúa Torre**.
Es una restricción de una sola variable. Utiliza una función anónima lambda para forzar a que el valor asignado a "grua_torre" sea estrictamente igual a "zona_estable". Usé el método **addConstraint()**.

3. Definí la restricción 2, **Excavadora**, de una sola variable. El motor descartará cualquier combinación donde la "excavadora" tenga asignado el valor "zona_estrecha". Usé el método **addConstraint()**.
4.   Definí la Restricción 3. Garantiza que las variables "hormigonera" y "grua_torre" tomen valores diferenetes. Usé el método **addConstraint()**.

5. Definí la Restricción 4, **Exclusividad**. Esta es una restricción global propia de la librería. Garantiza que todas las variables definidas en el problema tomen valores diferentes. Evita que dos máquinas terminen en la misma zona. Usé el método **addConstraint()**.


## Parámetros de Entrega y Evaluación
El entregable debe cumplir con lo siguiente:

* **Justificación Funcional:** Debes explicar en celdas de texto por qué el modelo de grafos y árboles es superior a una simple lista de if/else para este problema.
* **Documentación:** El código debe estar respaldado por una explicación de cómo opera el motor de inferencia en cada caso.

*Tener en cuenta que se evalua proceso y no resultado. Acordarse de justificar las elecciones en cada caso.*